# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn import preprocessing
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold


pd.set_option('display.max_columns', None)



df = pd.read_csv('willenscharen.csv', encoding='latin1', delimiter = ',', index_col = 0)
df = df.backfill(axis=0)
df = df.ffill(axis=0)

df = df.iloc[22:]
df

,bfwls,LDIz,LTPad,LFPad,QPad,WPad,NPad,QSarl,WSarl,QWill,ObereStoer,Buenzau,Aalbek,Willenscharen_pegel_cm,Willenscharen_radar_mm
tstamp,,,,,,,,,,,,,,,
2014-01-02 00:00:00,106,1000.100000,2.600000,90.000000,2.219302,34.0,0.0,3.118159,491.3002,6.912554,0.0,0.0,0.0,179.0,0.0
2014-01-02 01:00:00,106,999.000000,2.500000,90.000000,2.219516,34.0,0.0,3.086809,490.7000,6.796437,0.0,0.0,0.0,178.0,0.0
2014-01-02 02:00:00,106,998.400000,3.300000,89.000000,2.219943,34.0,0.0,3.086918,490.7000,6.679450,0.1,0.0,0.0,177.0,0.0
2014-01-02 03:00:00,106,997.500000,3.800000,87.000000,2.220371,34.0,0.0,3.102725,491.0000,6.679261,0.0,0.0,0.0,177.0,0.0
2014-01-02 04:00:00,106,996.900000,4.100000,87.000000,2.295369,35.0,0.0,3.113299,491.2000,6.679072,0.0,0.0,0.0,177.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-03-06 02:00:00,100,1024.100000,-0.908333,81.608333,3.754911,62.0,0.0,4.643617,518.0000,9.614127,0.0,0.0,0.0,207.0,0.0
2022-03-06 03:00:00,100,1023.966667,-0.650000,80.083333,3.754911,62.0,0.0,4.643617,518.0000,9.507003,0.0,0.0,0.0,206.0,0.0
2022-03-06 04:00:00,100,1023.983333,0.208333,71.200000,3.675090,61.0,0.0,4.643617,518.0000,9.399932,0.0,0.0,0.0,205.0,0.0


# Standardize


In [2]:
df_unstand = df

standardScaler = preprocessing.StandardScaler()
standardizedData = standardScaler.fit_transform(df)

df = pd.DataFrame(standardizedData, columns=df.columns, index=df.index)
df

,bfwls,LDIz,LTPad,LFPad,QPad,WPad,NPad,QSarl,WSarl,QWill,ObereStoer,Buenzau,Aalbek,Willenscharen_pegel_cm,Willenscharen_radar_mm
tstamp,,,,,,,,,,,,,,,
2014-01-02 00:00:00,0.540445,-1.152654,-1.031370,0.455491,0.055854,-0.198930,-0.195832,0.142520,0.279728,0.211747,-0.204285,-0.213547,-0.211022,0.180257,-0.222882
2014-01-02 01:00:00,0.540445,-1.261591,-1.045719,0.455491,0.055991,-0.198930,-0.195832,0.130745,0.265561,0.186939,-0.204285,-0.213547,-0.211022,0.153226,-0.222882
2014-01-02 02:00:00,0.540445,-1.321011,-0.930928,0.389272,0.056263,-0.198930,-0.195832,0.130785,0.265561,0.161944,0.029659,-0.213547,-0.211022,0.126196,-0.222882
2014-01-02 03:00:00,0.540445,-1.410141,-0.859184,0.256832,0.056535,-0.198930,-0.195832,0.136723,0.272642,0.161904,-0.204285,-0.213547,-0.211022,0.126196,-0.222882
2014-01-02 04:00:00,0.540445,-1.469562,-0.816138,0.256832,0.104273,-0.144214,-0.195832,0.140695,0.277363,0.161864,-0.204285,-0.213547,-0.211022,0.126196,-0.222882
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-03-06 02:00:00,0.337437,1.224156,-1.534776,-0.100204,1.033299,1.333107,-0.195832,0.715518,0.909977,0.788939,-0.204285,-0.213547,-0.211022,0.937117,-0.222882
2022-03-06 03:00:00,0.337437,1.210951,-1.497708,-0.201189,1.033299,1.333107,-0.195832,0.715518,0.909977,0.766052,-0.204285,-0.213547,-0.211022,0.910087,-0.222882
2022-03-06 04:00:00,0.337437,1.212602,-1.374547,-0.789442,0.982491,1.278392,-0.195832,0.715518,0.909977,0.743176,-0.204285,-0.213547,-0.211022,0.883056,-0.222882


# Sampling


### Convert to Array and create the samples as desired


In [3]:
three_d = 15*3*24

df_list = df.values.tolist()
df_list_flat_x = [cell for row in df_list for cell in row]
df_list_flat_y = df_list_flat_x[13::15]

df_x_arr = []
df_y_arr = []



for i in range(70000):
  df_x_arr.append(df_list_flat_x[i*15:i*15 + three_d])
### Convert to Array and create the features as desired

for i in range(70000):
  df_y_arr.append(df_list_flat_y[i + 72:i + 96])

### Convert back to pandas structure


In [4]:
df_x = pd.DataFrame(df_x_arr)
df_y = pd.DataFrame(df_y_arr)

### Restore the naming


In [5]:
df_x_rename = df_x
### Convert back to pandas structure

df_x_rename['date'] = df.iloc[0:70000].index
df_x_rename = df_x_rename.set_index(['date'])
df_x_rename.index = pd.to_datetime(df_x_rename.index)


for i in range(0, 1066, 15):
    df_x_rename = df_x_rename.rename(columns={
        df_x_rename.columns[i + 0]: "bfwls+" + str(i/15),
        df_x_rename.columns[i + 1]: "LDIz+" + str(i/15),
        df_x_rename.columns[i + 2]: "LTPad+" + str(i/15),
        df_x_rename.columns[i + 3]: "LFPad+" + str(i/15),
        df_x_rename.columns[i + 4]: "QPad+" + str(i/15),
        df_x_rename.columns[i + 5]: "WPad+" + str(i/15),
        df_x_rename.columns[i + 6]: "NPad+" + str(i/15),
        df_x_rename.columns[i + 7]: "QSarl+" + str(i/15),
        df_x_rename.columns[i + 8]: "WSarl+" + str(i/15),
        df_x_rename.columns[i + 9]: "QWill+" + str(i/15),
        df_x_rename.columns[i + 10]: "ObereStoer+" + str(i/15),
        df_x_rename.columns[i + 11]: "Buenzau+" + str(i/15),
        df_x_rename.columns[i + 12]: "Aalbek+" + str(i/15),
        df_x_rename.columns[i + 13]: "Willenscharen_pegel_cm+" + str(i/15),
        df_x_rename.columns[i + 14]: "Willenscharen_radar_mm+" + str(i/15)
           })
    
df_x_final = df_x_rename

In [6]:
df_y_rename = df_y

df_y_rename['date'] = df.iloc[0:70000].index
df_y_rename = df_y_rename.set_index(['date'])
df_y_rename.index = pd.to_datetime(df_y_rename.index)


df_y_final = df_y_rename

### Output the samples

In [7]:
df_x_final

bfwls+0.0  LDIz+0.0  LTPad+0.0  LFPad+0.0  QPad+0.0  \
date                                                                       
2014-01-02 00:00:00   0.540445 -1.152654  -1.031370   0.455491  0.055854   
2014-01-02 01:00:00   0.540445 -1.261591  -1.045719   0.455491  0.055991   
2014-01-02 02:00:00   0.540445 -1.321011  -0.930928   0.389272  0.056263   
2014-01-02 03:00:00   0.540445 -1.410141  -0.859184   0.256832  0.056535   
2014-01-02 04:00:00   0.540445 -1.469562  -0.816138   0.256832  0.104273   
...                        ...       ...        ...        ...       ...   
2021-12-30 20:00:00   0.574280 -0.413202   0.166759   0.895302 -0.004804   
2021-12-30 21:00:00   0.574280 -0.388443   0.163172   0.916823  0.038376   
2021-12-30 22:00:00   0.574280 -0.371938   0.163172   0.931723  0.081659   
2021-12-30 23:00:00   0.574280 -0.375239   0.170347   0.927860  0.124992   
2021-12-31 00:00:00   0.811123 -0.393395   0.188283   0.925101  0.124992   

                     WPad+0.0  NPad+0.0  QSarl+0.0  WSarl+0.0  QWill+0.0  \
date                                                                       
2014-01-02 00:00:00 -0.198930 -0.195832   0.142520   0.279728   0.211747   
2014-01-02 01:00:00 -0.198930 -0.195832   0.130745   0.265561   0.186939   
2014-01-02 02:00:00 -0.198930 -0.195832   0.130785   0.265561   0.161944   
2014-01-02 03:00:00 -0.198930 -0.195832   0.136723   0.272642   0.161904   
2014-01-02 04:00:00 -0.144214 -0.195832   0.140695   0.277363   0.161864   
...                       ...       ...        ...        ...        ...   
2021-12-30 20:00:00  0.074649  0.134725   0.134379   0.201827   0.046985   
2021-12-30 21:00:00  0.129364  0.112688   0.171442   0.249037   0.091577   
2021-12-30 22:00:00  0.184080  0.178800   0.208395   0.296247   0.113827   
2021-12-30 23:00:00  0.238795  0.950099   0.245330   0.343457   0.136053   
2021-12-31 00:00:00  0.238795  1.699362   0.282337   0.390667   0.180459   

                     ObereStoer+0.0  Buenzau+0.0  Aalbek+0.0  \
date                                                           
2014-01-02 00:00:00       -0.204285    -0.213547   -0.211022   
2014-01-02 01:00:00       -0.204285    -0.213547   -0.211022   
2014-01-02 02:00:00        0.029659    -0.213547   -0.211022   
2014-01-02 03:00:00       -0.204285    -0.213547   -0.211022   
2014-01-02 04:00:00       -0.204285    -0.213547   -0.211022   
...                             ...          ...         ...   
2021-12-30 20:00:00       -0.204285    -0.213547   -0.211022   
2021-12-30 21:00:00        0.029659     0.001572    0.016161   
2021-12-30 22:00:00       -0.204285    -0.213547   -0.211022   
2021-12-30 23:00:00       -0.204285     0.001572   -0.211022   
2021-12-31 00:00:00        0.965435     1.292290    1.379261   

                     Willenscharen_pegel_cm+0.0  Willenscharen_radar_mm+0.0  \
date                                                                          
2014-01-02 00:00:00                    0.180257                   -0.222882   
2014-01-02 01:00:00                    0.153226                   -0.222882   
2014-01-02 02:00:00                    0.126196                   -0.222882   
2014-01-02 03:00:00                    0.126196                   -0.222882   
2014-01-02 04:00:00                    0.126196                   -0.222882   
...                                         ...                         ...   
2021-12-30 20:00:00                    0.045103                   -0.222882   
2021-12-30 21:00:00                    0.099165                   -0.222882   
2021-12-30 22:00:00                    0.126196                   -0.222882   
2021-12-30 23:00:00                    0.153226                   -0.222882   
2021-12-31 00:00:00                    0.207288                   -0.222882   

                     bfwls+1.0  LDIz+1.0  LTPad+1.0  LFPad+1.0  QPad+1.0  \
date                                                                       
2014-01-02 00:00:00   0.540445

In [8]:
df_y_final

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
date,,,,,,,,,,,,,,,,,,,,,,,,
2014-01-02 00:00:00,0.315411,0.315411,0.315411,0.315411,0.288380,0.288380,0.288380,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.234318,0.234318,0.234318,0.234318,0.234318,0.234318,0.207288,0.207288
2014-01-02 01:00:00,0.315411,0.315411,0.315411,0.288380,0.288380,0.288380,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.234318,0.234318,0.234318,0.234318,0.234318,0.234318,0.207288,0.207288,0.207288
2014-01-02 02:00:00,0.315411,0.315411,0.288380,0.288380,0.288380,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.234318,0.234318,0.234318,0.234318,0.234318,0.234318,0.207288,0.207288,0.207288,0.207288
2014-01-02 03:00:00,0.315411,0.288380,0.288380,0.288380,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.234318,0.234318,0.234318,0.234318,0.234318,0.234318,0.207288,0.207288,0.207288,0.207288,0.207288
2014-01-02 04:00:00,0.288380,0.288380,0.288380,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.261349,0.234318,0.234318,0.234318,0.234318,0.234318,0.234318,0.207288,0.207288,0.207288,0.207288,0.207288,0.207288
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-30 20:00:00,2.694114,2.775206,2.829268,2.856298,2.910360,2.937391,2.964421,2.991452,3.018483,3.045513,3.072544,3.072544,3.099575,3.126606,3.153636,3.180667,3.207698,3.234728,3.234728,3.288790,3.288790,3.288790,3.288790,3.315821
2021-12-30 21:00:00,2.775206,2.829268,2.856298,2.910360,2.937391,2.964421,2.991452,3.018483,3.045513,3.072544,3.072544,3.099575,3.126606,3.153636,3.180667,3.207698,3.234728,3.234728,3.288790,3.288790,3.288790,3.288790,3.315821,3.315821
2021-12-30 22:00:00,2.829268,2.856298,2.910360,2.937391,2.964421,2.991452,3.018483,3.045513,3.072544,3.072544,3.099575,3.126606,3.153636,3.180667,3.207698,3.234728,3.234728,3.288790,3.288790,3.288790,3.288790,3.315821,3.315821,3.315821


# PCA

### Creating a PCA with 78 components (so the explained variance is greater then 90%)
### Explained Variance with 2 Components is only ~40%!

In [ ]:
from sklearn.decomposition import PCA as sklearnPCA
pca = sklearnPCA(n_components=78)

model = pca.fit(df_x_final)
pcaTransformedData = model.transform(df_x_final)

### Explained Variance of each Component and Sum

In [ ]:
pca.explained_variance_ratio_

In [ ]:
pca.explained_variance_ratio_.sum()

### Create pandas showing the impact of the features of the first two components and plot some of them

In [ ]:
feature_names = list(df_x_final)
feature_names[0] = list(df_x_final)
feature_names[1] = [abs(ele) for ele in pca.components_[0].tolist()]
feature_names[2] = [abs(ele) for ele in pca.components_[1].tolist()]



df_features = pd.DataFrame(feature_names)
df_features = df_features.iloc[0:3].T

df_features = df_features.set_index([df_features.columns[0]])

matplotlib.rc('figure', figsize=(20, 5), dpi=300)
plt.xticks(rotation=90)

i = 1
plt.plot(df_features[df_features.columns[0]].iloc[i*15:i*15 + 15])
plt.show()

df_features.dtypes
df_features.to_csv('pca_feature.csv')



### Features with high impact on Component 1 are: Abfluss/Wasserstand aller Stationen
### Features with some impact on Component 1 are: Bodenfeuchte, Luftdruck, Lufttemperatur, Luftfeuchte
### Features with low impact on Component 1 are: Niederschlag generell



### Plot first 20000 values for Component 1 and 2 and compare with our Sample Target Y

In [ ]:
#pcaTransformedFrame = pd.DataFrame(pcaTransformedData, columns=["PCA1", "PCA2"])
#pcaTransformedFrame.iloc[0:20000].plot()

pcaTransformedFrame = pd.DataFrame(pcaTransformedData[: , :2], columns=["PCA1", "PCA2"])
pcaTransformedFrame.iloc[0:20000].plot()


In [ ]:
plt.xticks(rotation=90)
plt.plot(df_y_final.iloc[0:20000][df_y_final.columns[0]])
plt.show()

# Feature Selection with Correllation Matrix

### Add Y to to X and calculate Correlation Matrix to filter out irrelevant Features

In [ ]:
df_xy = df_x_final.join(df_y_final)
df_corr_y = df_xy.corr()[:-24]
df_corr_y

### Show average correlation of each feature with Y

In [ ]:
average_corr_y = df_corr_y.iloc[:,1080:1104].mean(axis=1)
average_corr_y

### Filter Features with average correlation of less then 0.2 (not very relevant)

In [ ]:
average_corr_y_filtered = average_corr_y[average_corr_y.abs() < 0.2].dropna()
average_corr_y_filtered

In [ ]:
df_x_drop_irr = df_x_final.drop(df_x_final[average_corr_y_filtered.index], axis=1)
df_x_drop_irr

### Create Correlation Matrix for remaining features to filter out redundant ones

In [ ]:
df_corr = df_x_drop_irr.corr()

In [ ]:
df_corr

### Extract Features with correlation greater then 0.8 to each other (redundant)

In [ ]:
highly_correlated = set()

for i in range(len(df_corr.columns)):### Create Correlation Matrix (which shows correlation between each feature)
    for j in range(i):
        if abs(df_corr.iloc[i, j]) > 0.8:
            col = df_corr.columns[i]
            highly_correlated.add(col)
        
highly_correlated


### Drop one each of those Features

In [ ]:
df_features = df_x_drop_irr.drop(df_corr[highly_correlated], axis=1)

In [ ]:
df_features

In [ ]:
df_features.join(df_y_final).corr()[:-24]